To execute this notebook, you need to either
 - Download a pre-trained model
 - Train an example model by excecuting the model_training_tutorial.ipynb notebook

In this tutorial we will learn to:
- Load a previously trained model
- Extract DeepPrint features from fingerprint images
- Evaluate the performance of the extracted fixed-length representations

## Embedding extraction

After training the model, we can extract the DeepPrint features for the fingerprint images. This is done by calling the `extract` method of the `DeepPrintExtractor` class.

In [2]:
import os

from flx.extractor.fixed_length_extractor import get_DeepPrint_Tex, get_DeepPrint_TexMinu, DeepPrintExtractor

# Dimension and number of training subjects must be known to load the pre-trained model

# To load the pre-trained model parameters use num_training_subjects=8000
extractor: DeepPrintExtractor = get_DeepPrint_TexMinu(num_training_subjects=8000, num_dims=256)

# To load the pre-trained model parameters use
MODEL_DIR: str = os.path.abspath(".") # Path to the directory containing the model parameters
extractor.load_best_model(MODEL_DIR)

Loaded best model from /home/sarthak/best_model.pyt


Now we need to specify the dataset, for which we want to extract the embeddings

In [4]:
import os

from flx.data.dataset import *
from flx.data.image_loader import SFingeLoader
from flx.data.transformed_image_loader import TransformedImageLoader
from flx.image_processing.binarization import LazilyAllocatedBinarizer
from flx.data.image_helpers import pad_and_resize_to_deepprint_input_size

# NOTE: If this does not work, enter the absolute path to the notebooks/example-dataset directory here! 
DATASET_PATH: str = os.path.abspath("./example-dataset")

# We will use the SFingeLoader to load the images from the dataset
image_loader = TransformedImageLoader(
        images=SFingeLoader(DATASET_PATH),
        poses=None,
        transforms=[
            LazilyAllocatedBinarizer(5.0),
            pad_and_resize_to_deepprint_input_size,
        ],
    )

image_dataset: Dataset = Dataset(image_loader, image_loader.ids)

# The second value is for the minutiae branch, which we do not have in this example
texture_embeddings, minutia_embeddings = extractor.extract(image_dataset)

Created IdentifierSet with 1 subjects and a total of 2 samples.


100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.09s/it]


In [9]:
print(texture_embeddings.numpy())
print(minutia_embeddings.numpy())
print(len(texture_embeddings.numpy()[1]))
print(len(minutia_embeddings.numpy()[1]))

[[-4.95610163e-02 -1.72470305e-02 -4.20934381e-03 -4.24728170e-02
  -2.70883702e-02 -6.60380796e-02 -2.58858483e-02 -7.27722375e-03
   6.46459013e-02  1.55428588e-01 -5.21242283e-02 -3.40207666e-02
   9.40556452e-02  2.22103558e-02 -1.02857612e-01 -4.30838503e-02
  -1.88429728e-02 -5.21410480e-02 -6.11949451e-02 -2.97304355e-02
  -7.98685774e-02 -3.53166461e-02 -2.49366164e-02  1.09952919e-01
   6.36244565e-02 -3.61787677e-02  2.86442116e-02 -7.27332905e-02
  -2.81174220e-02  4.35304008e-02  3.49379443e-02 -1.07531948e-02
   1.03204295e-01  3.54664959e-03 -3.91707616e-03 -5.39970472e-02
   5.63578568e-02 -5.43844923e-02  1.05654700e-02 -2.21341513e-02
  -5.40021323e-02  7.86144137e-02  4.18705121e-02 -2.80573517e-02
   5.29733524e-02 -3.44438069e-02 -2.88897604e-02 -4.15910818e-02
  -9.31501910e-02  2.79882122e-02  1.07782982e-01 -8.89791525e-04
  -9.90459044e-03 -9.15734619e-02  9.85097513e-02 -2.91763414e-02
  -5.45528680e-02 -6.60348125e-03 -1.58563480e-02  1.73279084e-02
  -6.99609

## Benchmarking

To evaluate the embeddings, we want to run a benchmark on them. For this, we must first specify the type of benchmark, and which comparisons should be run.

In [ ]:
from flx.scripts.generate_benchmarks import create_verification_benchmark

NUM_IMPRESSIONS_PER_SUBJECT = 10
benchmark = create_verification_benchmark(
    subjects=list(range(image_dataset.num_subjects)),
    impressions_per_subject=list(range(NUM_IMPRESSIONS_PER_SUBJECT))
)

Now we can run the benchmark. To do this, we must first specify the matcher (in our case cosine similarity of the embeddings)

In [ ]:
from flx.benchmarks.matchers import CosineSimilarityMatcher
from flx.data.embedding_loader import EmbeddingLoader

# We concatenate texture and minutia embedding vectors
embeddings = EmbeddingLoader.combine(texture_embeddings, minutia_embeddings)
matcher = CosineSimilarityMatcher(EmbeddingLoader.combine(texture_embeddings, minutia_embeddings))

results = benchmark.run(matcher)

print(f"Equal-Error-Rate: {results.get_equal_error_rate()}")

To visualize the results, we can plot a DET curve. (Do not wonder if it is empty, probably the model is not trained enough. Take a look at the EER instead.)

In [ ]:
from flx.visualization.plot_DET_curve import plot_verification_results

figure_path = "DET_curve"

# Lists are used to allow for multiple models to be plotted in the same figure
plot_verification_results(figure_path, results=[results], model_labels=["DeepPrint_TexMinu"], plot_title="example-dataset - verification")